In [1]:
from pathlib import Path
import json

# Path to the dataset root
DATA_ROOT = Path("../data/imagenet100")  # relative to the notebooks/ folder

# Sanity: does the path exist?
assert DATA_ROOT.exists(), f"Data root not found at {DATA_ROOT.resolve()}"

# Load the labels file (synset_id -> human-readable class name)
with open(DATA_ROOT / "Labels.json") as f:
    labels = json.load(f)

print(f"Number of classes in Labels.json: {len(labels)}")
print(f"\nFirst 5 entries:")
for synset, name in list(labels.items())[:5]:
    print(f"  {synset}: {name}")

Number of classes in Labels.json: 100

First 5 entries:
  n01968897: chambered nautilus, pearly nautilus, nautilus
  n01770081: harvestman, daddy longlegs, Phalangium opilio
  n01818515: macaw
  n02011460: bittern
  n01496331: electric ray, crampfish, numbfish, torpedo


In [2]:
def collect_files(split_folders):
    """Walk one or more folders and collect (path, synset_id) tuples."""
    items = []
    for folder in split_folders:
        if not folder.exists():
            print(f"Warning: {folder} does not exist, skipping")
            continue
        for class_dir in sorted(folder.iterdir()):
            if not class_dir.is_dir():
                continue
            synset = class_dir.name
            for img_path in class_dir.iterdir():
                if img_path.suffix.upper() == ".JPEG":
                    items.append((img_path, synset))
    return items


train_folders = [DATA_ROOT / f"train.X{i}" for i in [1, 2, 3, 4]]
train_items = collect_files(train_folders)

heldout_items = collect_files([DATA_ROOT / "val.X"])

print(f"Train images: {len(train_items):,}")
print(f"Held-out images: {len(heldout_items):,}")
print(f"\nFirst 3 train entries:")
for path, synset in train_items[:3]:
    print(f"  {synset:12} -> {path.name}")

Train images: 130,000
Held-out images: 5,000

First 3 train entries:
  n01440764    -> n01440764_32420.JPEG
  n01440764    -> n01440764_6949.JPEG
  n01440764    -> n01440764_22135.JPEG


In [3]:
train_synsets = sorted(set(synset for _, synset in train_items))
heldout_synsets = sorted(set(synset for _, synset in heldout_items))
labels_synsets = sorted(labels.keys())

assert train_synsets == heldout_synsets, "Train and held-out have different classes!"
assert train_synsets == labels_synsets, "Synsets in folders don't match Labels.json!"

print(f"All three sources agree: 100 classes")

synset_to_idx = {synset: i for i, synset in enumerate(train_synsets)}
idx_to_synset = {i: synset for synset, i in synset_to_idx.items()}
idx_to_name = {i: labels[synset] for synset, i in synset_to_idx.items()}

print(f"\nFirst 5 classes (idx -> synset -> name):")
for i in range(5):
    synset = idx_to_synset[i]
    print(f"  {i:3} -> {synset} -> {idx_to_name[i]}")

All three sources agree: 100 classes

First 5 classes (idx -> synset -> name):
    0 -> n01440764 -> tench, Tinca tinca
    1 -> n01443537 -> goldfish, Carassius auratus
    2 -> n01484850 -> great white shark, white shark, man-eater, man-eating shark, Carcharodon carcharias
    3 -> n01491361 -> tiger shark, Galeocerdo cuvieri
    4 -> n01494475 -> hammerhead, hammerhead shark


In [4]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import torch

# Standard ImageNet preprocessing pipeline:
# 1. Resize so shorter side = 256 (preserves aspect ratio)
# 2. Center-crop 224x224 from the middle
# 3. Convert PIL image to PyTorch tensor in [0, 1]
# 4. Normalize with ImageNet mean/std (Barlow Twins was trained on these stats)
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

print("Preprocess transform defined")
print(preprocess)

Preprocess transform defined
Compose(
    Resize(size=256, interpolation=bilinear, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


In [5]:
class ImageNet100Dataset(Dataset):
    """
    Loads ImageNet-100 images and returns (preprocessed_tensor, class_index) tuples.
    
    Args:
        items: list of (image_path, synset_id) tuples
        synset_to_idx: dict mapping synset_id (str) to class index (int)
        transform: optional torchvision transform to apply (e.g. preprocess pipeline)
    """
    def __init__(self, items, synset_to_idx, transform=None):
        self.items = items
        self.synset_to_idx = synset_to_idx
        self.transform = transform
    
    def __len__(self):
        return len(self.items)
    
    def __getitem__(self, idx):
        img_path, synset = self.items[idx]
        # Load image. .convert("RGB") handles grayscale or RGBA images uniformly
        # (rare in ImageNet but present — e.g. some grayscale photos)
        image = Image.open(img_path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        label = self.synset_to_idx[synset]
        return image, label


# Build the train and held-out datasets
train_dataset = ImageNet100Dataset(train_items, synset_to_idx, transform=preprocess)
heldout_dataset = ImageNet100Dataset(heldout_items, synset_to_idx, transform=preprocess)

print(f"Train dataset: {len(train_dataset):,} samples")
print(f"Held-out dataset: {len(heldout_dataset):,} samples")

Train dataset: 130,000 samples
Held-out dataset: 5,000 samples


In [6]:
# Pull a single sample
image_tensor, label = train_dataset[0]

print(f"Image tensor shape: {image_tensor.shape}")
print(f"Image tensor dtype: {image_tensor.dtype}")
print(f"Image tensor device: {image_tensor.device}")
print(f"Pixel value range: [{image_tensor.min():.3f}, {image_tensor.max():.3f}]")
print(f"Per-channel mean: {image_tensor.mean(dim=(1,2)).tolist()}")
print(f"Label: {label}")
print(f"Class name: {idx_to_name[label]}")

Image tensor shape: torch.Size([3, 224, 224])
Image tensor dtype: torch.float32
Image tensor device: cpu
Pixel value range: [-2.101, 2.640]
Per-channel mean: [-0.7029762864112854, -0.5683143138885498, -0.5009427666664124]
Label: 0
Class name: tench, Tinca tinca


In [28]:
# Build DataLoaders
# - batch_size: how many images per batch. 64 is a reasonable default for embedding extraction
# - shuffle=True for training, False for held-out (order doesn't matter when evaluating)
# - num_workers: parallel processes that load images. M3 Air has 8 cores; 4 is conservative
# - persistent_workers: keeps workers alive between epochs (saves startup overhead)
# - pin_memory: speeds up CPU→GPU transfers; works for MPS too

BATCH_SIZE = 256

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,          
    pin_memory=False,
)

heldout_loader = DataLoader(
    heldout_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,           
    pin_memory=False,
)

print(f"Train batches per epoch: {len(train_loader):,}")
print(f"Held-out batches: {len(heldout_loader):,}")

Train batches per epoch: 458
Held-out batches: 20


In [15]:
import time

# Pull one batch and time it
start = time.time()
batch_iter = iter(train_loader)
images, labels = next(batch_iter)
elapsed = time.time() - start

print(f"Batch loaded in {elapsed:.2f} seconds")
print(f"Images tensor shape: {images.shape}")
print(f"Images dtype: {images.dtype}")
print(f"Labels tensor shape: {labels.shape}")
print(f"Labels dtype: {labels.dtype}")
print(f"\nFirst 10 labels: {labels[:10].tolist()}")
print(f"Pixel value range across batch: [{images.min():.3f}, {images.max():.3f}]")

Batch loaded in 0.20 seconds
Images tensor shape: torch.Size([64, 3, 224, 224])
Images dtype: torch.float32
Labels tensor shape: torch.Size([64])
Labels dtype: torch.int64

First 10 labels: [71, 10, 87, 36, 21, 23, 53, 82, 67, 48]
Pixel value range across batch: [-2.118, 2.640]


In [16]:
times = []
for i in range(5):
    start = time.time()
    images, labels = next(batch_iter)
    elapsed = time.time() - start
    times.append(elapsed)
    print(f"Batch {i+1}: {elapsed:.3f}s")

print(f"\nMean batch fetch time: {sum(times)/len(times):.3f}s")

Batch 1: 0.203s
Batch 2: 0.159s
Batch 3: 0.164s
Batch 4: 0.158s
Batch 5: 0.172s

Mean batch fetch time: 0.171s


In [17]:
from sklearn.model_selection import train_test_split

# Extract just the integer labels (we'll reuse train_items as the source of truth)
train_labels_for_split = [synset_to_idx[synset] for _, synset in train_items]

# Stratified 90/10 split
# random_state=42 ensures reproducibility — the same split every time we run
train_subset, val_subset, _, _ = train_test_split(
    train_items,
    train_labels_for_split,
    test_size=0.10,            # 10% for validation
    stratify=train_labels_for_split,
    random_state=42,
)

print(f"Train subset: {len(train_subset):,}")
print(f"Validation subset: {len(val_subset):,}")
print(f"Held-out: {len(heldout_items):,}")

Train subset: 117,000
Validation subset: 13,000
Held-out: 5,000


In [18]:
from collections import Counter

# Count classes in each subset
train_class_counts = Counter(synset_to_idx[s] for _, s in train_subset)
val_class_counts = Counter(synset_to_idx[s] for _, s in val_subset)

# All 100 classes should be in both splits
assert len(train_class_counts) == 100, f"Train has only {len(train_class_counts)} classes!"
assert len(val_class_counts) == 100, f"Validation has only {len(val_class_counts)} classes!"

# Check the count distribution
train_counts = sorted(train_class_counts.values())
val_counts = sorted(val_class_counts.values())

print(f"Train: min={train_counts[0]}, max={train_counts[-1]}, mean={sum(train_counts)/100:.1f}")
print(f"Val:   min={val_counts[0]}, max={val_counts[-1]}, mean={sum(val_counts)/100:.1f}")
print(f"\nExpected: train ~1170 per class, val ~130 per class")

Train: min=1170, max=1170, mean=1170.0
Val:   min=130, max=130, mean=130.0

Expected: train ~1170 per class, val ~130 per class


In [19]:
# Build datasets for the three splits
train_dataset = ImageNet100Dataset(train_subset, synset_to_idx, transform=preprocess)
val_dataset = ImageNet100Dataset(val_subset, synset_to_idx, transform=preprocess)
heldout_dataset = ImageNet100Dataset(heldout_items, synset_to_idx, transform=preprocess)

print(f"Train:    {len(train_dataset):>7,} samples")
print(f"Val:      {len(val_dataset):>7,} samples")
print(f"Held-out: {len(heldout_dataset):>7,} samples")
print(f"Total:    {len(train_dataset) + len(val_dataset) + len(heldout_dataset):>7,} samples")

Train:    117,000 samples
Val:       13,000 samples
Held-out:   5,000 samples
Total:    135,000 samples


In [20]:
import torch.nn as nn

# Set the device — MPS on Apple Silicon
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [21]:
# Load Barlow Twins pretrained ResNet-50 from PyTorch Hub
# First call downloads ~100MB; subsequent calls use the cached copy
print("Loading Barlow Twins pretrained ResNet-50...")
backbone = torch.hub.load('facebookresearch/barlowtwins:main', 'resnet50')
print("Loaded.")

# Replace the final FC layer (which maps 2048 -> 1000 ImageNet classes) with Identity
# After this, backbone(images) returns the raw 2048-d encoder features
backbone.fc = nn.Identity()

# Freeze all parameters — we'll never update the backbone
for param in backbone.parameters():
    param.requires_grad = False

# Set to evaluation mode — disables dropout and uses running statistics for BatchNorm
backbone.eval()

# Move to device
backbone = backbone.to(device)

# Count parameters as a sanity check
total_params = sum(p.numel() for p in backbone.parameters())
trainable_params = sum(p.numel() for p in backbone.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Loading Barlow Twins pretrained ResNet-50...
Downloading: "https://github.com/facebookresearch/barlowtwins/zipball/main" to /Users/ayushkumarsingh/.cache/torch/hub/main.zip


/opt/homebrew/Caskroom/miniconda/base/envs/tversky/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/tversky/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Downloading: "https://dl.fbaipublicfiles.com/barlowtwins/ep1000_bs2048_lrw0.2_lrb0.0048_lambd0.0051/resnet50.pth" to /Users/ayushkumarsingh/.cache/torch/hub/checkpoints/resnet50.pth


100%|██████████| 90.0M/90.0M [00:04<00:00, 22.0MB/s]


Loaded.
Total parameters: 23,508,032
Trainable parameters: 0


In [22]:
# Pull one batch from the train loader
images, labels = next(iter(train_loader))
print(f"Input batch shape: {images.shape}")
print(f"Input batch device: {images.device}")

# Move images to MPS
images = images.to(device)

# Forward pass with no_grad — we're not training, don't track gradients
import time
with torch.no_grad():
    start = time.time()
    embeddings = backbone(images)
    # MPS is asynchronous; sync to get accurate timing
    if device.type == "mps":
        torch.mps.synchronize()
    elapsed = time.time() - start

print(f"\nOutput embedding shape: {embeddings.shape}")
print(f"Output device: {embeddings.device}")
print(f"Output dtype: {embeddings.dtype}")
print(f"Forward pass time: {elapsed:.3f} seconds")
print(f"\nEmbedding stats:")
print(f"  Mean: {embeddings.mean():.4f}")
print(f"  Std:  {embeddings.std():.4f}")
print(f"  Min:  {embeddings.min():.4f}")
print(f"  Max:  {embeddings.max():.4f}")

Input batch shape: torch.Size([64, 3, 224, 224])
Input batch device: cpu

Output embedding shape: torch.Size([64, 2048])
Output device: mps:0
Output dtype: torch.float32
Forward pass time: 1.537 seconds

Embedding stats:
  Mean: 0.0522
  Std:  0.1096
  Min:  0.0000
  Max:  3.0229


In [23]:
# Time several more forward passes to see steady-state speed
times = []
for i in range(5):
    images, _ = next(iter(train_loader))
    images = images.to(device)
    start = time.time()
    with torch.no_grad():
        embeddings = backbone(images)
    if device.type == "mps":
        torch.mps.synchronize()
    elapsed = time.time() - start
    times.append(elapsed)
    print(f"Forward pass {i+1}: {elapsed:.3f}s ({64/elapsed:.1f} images/sec)")

mean_time = sum(times) / len(times)
print(f"\nMean: {mean_time:.3f}s per batch of 64")
print(f"Throughput: {64/mean_time:.1f} images/second")
print(f"Estimated time for 135,000 images: {135000/(64/mean_time)/60:.1f} minutes")

Forward pass 1: 0.533s (120.0 images/sec)
Forward pass 2: 0.478s (133.8 images/sec)
Forward pass 3: 0.432s (148.1 images/sec)
Forward pass 4: 0.425s (150.6 images/sec)
Forward pass 5: 0.425s (150.7 images/sec)

Mean: 0.459s per batch of 64
Throughput: 139.5 images/second
Estimated time for 135,000 images: 16.1 minutes


In [26]:
import sys
from pathlib import Path

# Find the project root by walking up until we see a known marker folder
def find_project_root(marker="data/imagenet100"):
    current = Path.cwd()
    for parent in [current, *current.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find project root containing {marker}")

PROJECT_ROOT = find_project_root()
print(f"Project root: {PROJECT_ROOT}")

# Confirm the src folder is where we expect
print(f"src/ folder exists: {(PROJECT_ROOT / 'src').exists()}")
print(f"src/dataset.py exists: {(PROJECT_ROOT / 'src' / 'dataset.py').exists()}")
print(f"src/__init__.py exists: {(PROJECT_ROOT / 'src' / '__init__.py').exists()}")

# Add project root to sys.path so 'from src.dataset import ...' works
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

Project root: /Users/ayushkumarsingh/ABNS-Project/tversky-asymmetric
src/ folder exists: True
src/dataset.py exists: True
src/__init__.py exists: True


In [27]:
from src.dataset import ImageNet100Dataset, IMAGENET_PREPROCESS

train_dataset = ImageNet100Dataset(train_subset, synset_to_idx, transform=IMAGENET_PREPROCESS)
val_dataset = ImageNet100Dataset(val_subset, synset_to_idx, transform=IMAGENET_PREPROCESS)
heldout_dataset = ImageNet100Dataset(heldout_items, synset_to_idx, transform=IMAGENET_PREPROCESS)

print(f"Train:    {len(train_dataset):>7,} samples")
print(f"Val:      {len(val_dataset):>7,} samples")
print(f"Held-out: {len(heldout_dataset):>7,} samples")

Train:    117,000 samples
Val:       13,000 samples
Held-out:   5,000 samples


In [29]:
# Pull a few batches at the new size and time them
times = []
batch_iter = iter(train_loader)
for i in range(5):
    images, _ = next(batch_iter)
    images = images.to(device)
    start = time.time()
    with torch.no_grad():
        embeddings = backbone(images)
    if device.type == "mps":
        torch.mps.synchronize()
    elapsed = time.time() - start
    times.append(elapsed)
    print(f"Forward pass {i+1}: {elapsed:.3f}s ({BATCH_SIZE/elapsed:.1f} images/sec)")

mean_time = sum(times) / len(times)
print(f"\nMean: {mean_time:.3f}s per batch of {BATCH_SIZE}")
print(f"Throughput: {BATCH_SIZE/mean_time:.1f} images/second")
print(f"Estimated time for 135,000 images: {135000/(BATCH_SIZE/mean_time)/60:.1f} minutes")


Forward pass 1: 2.295s (111.5 images/sec)
Forward pass 2: 1.777s (144.1 images/sec)
Forward pass 3: 1.760s (145.5 images/sec)
Forward pass 4: 1.748s (146.4 images/sec)
Forward pass 5: 1.746s (146.6 images/sec)

Mean: 1.865s per batch of 256
Throughput: 137.3 images/second
Estimated time for 135,000 images: 16.4 minutes
